# Codex Python SDK：原生 module 與 I/O

使用官方 `openai_codex` 的非同步介面。Notebook 直接啟動 runtime、建立 thread 與執行 turn。

| 原生 API | 輸入 | 輸出 |
|---|---|---|
| `CodexConfig` | runtime 環境與 TOML 覆寫 | 啟動設定 |
| `AsyncCodex.thread_start` | 模型、工作目錄、沙箱設定 | `AsyncThread` |
| `AsyncThread.run` | `str` 或 SDK 輸入項目清單 | `TurnResult` |
| `AsyncThread.turn` | `str` 或 SDK 輸入項目清單 | `AsyncTurnHandle` |
| `AsyncTurnHandle.stream` | 無額外輸入 | `Notification` 的非同步迭代器 |

`TurnResult` 是 dataclass。它不是 LangGraph 的狀態字典。
`Notification` 包含事件方法名稱與原生 payload。

本檔已保存一次真實執行的輸出。輸出中的識別碼、時間與用量只代表該次執行。


### 類別關係圖

實線表示設定、建立或回傳關係。`TurnResult` 是 dataclass；`Notification` 是事件資料，兩者的欄位不同。

執行下方區塊可顯示圖形。此區塊不會呼叫 LLM。


In [1]:
from IPython.display import HTML, display

display(HTML(r"""
<section style="background:#f8fafc;color:#172b4d;border:1px solid #dce4ef;border-radius:16px;padding:18px;margin:12px 0;font-family:system-ui,sans-serif;overflow-x:auto">
<h3 style="color:#172b4d;margin:0 0 12px;font-size:20px">類別關係圖</h3>
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1080 505" role="img" aria-label="Codex SDK 類別關係：設定、執行緒、結果與串流通知" style="display:block;width:100%;min-width:760px;height:auto;font-family:'Noto Sans CJK TC','Microsoft JhengHei',system-ui,sans-serif">
 <title>Codex SDK 類別關係：設定、執行緒、結果與串流通知</title><defs><marker id="codex-sdk-classes-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#64748b"/></marker></defs>
 <rect width="1080" height="505" fill="#f8fafc"/><rect x="25" y="25" width="290" height="158" rx="12" fill="#fff" stroke="#2563eb" stroke-width="1.6"/><path d="M26 73 H314" stroke="#dce4ef"/><text x="41" y="54" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">CodexConfig</text><text x="41" y="96" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">openai_codex</text><text x="41" y="125" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">cwd / env</text><text x="41" y="147" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">config_overrides</text><rect x="395" y="25" width="290" height="158" rx="12" fill="#fff" stroke="#2563eb" stroke-width="1.6"/><path d="M396 73 H684" stroke="#dce4ef"/><text x="411" y="54" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">AsyncCodex</text><text x="411" y="96" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">openai_codex</text><text x="411" y="125" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">啟動 / 關閉 app-server</text><text x="411" y="147" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">thread_start() → AsyncThread</text><rect x="765" y="25" width="290" height="158" rx="12" fill="#fff" stroke="#2563eb" stroke-width="1.6"/><path d="M766 73 H1054" stroke="#dce4ef"/><text x="781" y="54" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">AsyncThread</text><text x="781" y="96" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">openai_codex</text><text x="781" y="125" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">run(input) → TurnResult</text><text x="781" y="147" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">turn(input) → AsyncTurnHandle</text><rect x="25" y="310" width="290" height="158" rx="12" fill="#fff" stroke="#087e8b" stroke-width="1.6"/><path d="M26 358 H314" stroke="#dce4ef"/><text x="41" y="339" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">TurnResult</text><text x="41" y="381" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">openai_codex · dataclass</text><text x="41" y="410" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">status / final_response</text><text x="41" y="432" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">items / usage</text><rect x="395" y="310" width="290" height="158" rx="12" fill="#fff" stroke="#087e8b" stroke-width="1.6"/><path d="M396 358 H684" stroke="#dce4ef"/><text x="411" y="339" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">AsyncTurnHandle</text><text x="411" y="381" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">openai_codex</text><text x="411" y="410" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">stream() → 通知迭代器</text><text x="411" y="432" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">id / thread_id</text><rect x="765" y="310" width="290" height="158" rx="12" fill="#fff" stroke="#087e8b" stroke-width="1.6"/><path d="M766 358 H1054" stroke="#dce4ef"/><text x="781" y="339" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">Notification</text><text x="781" y="381" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">SDK 原生通知物件</text><text x="781" y="410" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">method: 事件方法名稱</text><text x="781" y="432" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">payload: 事件資料</text><polyline points="315,104 395,104" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-classes-arrow)"/><text x="355.0" y="90" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">設定</text><polyline points="685,104 765,104" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-classes-arrow)"/><text x="725.0" y="90" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">建立</text><polyline points="805,183 805,230 170,230 170,310" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-classes-arrow)"/><text x="290" y="217" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">run() 回傳</text><polyline points="910,183 910,268 540,268 540,310" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-classes-arrow)"/><text x="700" y="255" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">turn() 回傳</text><polyline points="685,389 765,389" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-classes-arrow)"/><text x="725.0" y="375" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">stream()</text></svg>
<p style="color:#52647c;line-height:1.7;margin:10px 0 0;font-size:14px">實線表示設定、建立或回傳關係。TurnResult 是 dataclass；Notification 是事件資料，兩者的欄位不同。</p>
</section>
"""))


## 1. 啟動與設定

在本資料夾執行：

```bash
uv sync --locked --group notebook
uv run --group notebook jupyter lab tutorial_native.ipynb
```

依序執行各區塊。推論與串流區塊會呼叫真實模型，並使用模型額度。
本教學直接使用頂層 `await`。每個串流範例會啟動另一個任務。
所有原生輸入與輸出都保留在 Notebook 變數中。


In [1]:
import os
import sys
from importlib.metadata import version
from pathlib import Path
from pprint import pprint

from dotenv import dotenv_values

assert sys.version_info[:2] == (3, 13), "請使用 Python 3.13 核心。"
project_dir = Path.cwd()
assert (project_dir / "pyproject.toml").is_file(), "請從本專案資料夾啟動 Notebook。"

# 直接載入設定。環境變數優先於專案 .env，再優先於根目錄 .env。
env = {
    **dotenv_values(project_dir.parent / ".env"),
    **dotenv_values(project_dir / ".env"),
    **os.environ,
}
endpoint = env["LITELLM_ENDPOINT"].strip().rstrip("/")
api_key = env["LITELLM_API_KEY"]
model_name = env.get("CODEX_MODEL") or env["LITELLM_MODEL"]
assert endpoint.startswith(("http://", "https://")), "端點需要 HTTP 或 HTTPS 協定。"
assert api_key and model_name, "請設定金鑰與模型別名。"
print("Python：", sys.version.split()[0])
print("模型：", model_name)
# 請勿顯示 env、api_key 或含有金鑰的 client 設定。

Python： 3.13.15
模型： gpt-5.4-mini


## 2. 使用公開 module

下列 import 都來自官方 SDK 的公開入口。
`TurnResult` 的實作 module 可能顯示為內部路徑。請使用 `openai_codex` 匯入它。


In [2]:
import json
from dataclasses import fields

from openai_codex import (
    ApprovalMode,
    AsyncCodex,
    CodexConfig,
    Sandbox,
    TextInput,
    TurnResult,
)

print("openai-codex：", version("openai-codex"))
print("TurnResult 欄位：")
for field in fields(TurnResult):
    print(field.name, field.type)

openai-codex： 0.163.0
TurnResult 欄位：
id str
status TurnStatus
error TurnError | None
started_at int | None
completed_at int | None
duration_ms int | None
final_response str | None
items list[ThreadItem]
usage ThreadTokenUsage | None


## 3. 直接設定 LiteLLM 供應者

Codex 使用 Responses API。LiteLLM 必須支援對應的串流事件。
`config_overrides` 接受 TOML 設定字串。`json.dumps` 用來正確引用網址。
金鑰由子程序環境傳入，不放在命令列參數中。
請勿顯示整個 `config`，因為 `env` 含有金鑰。


In [3]:
workspace = project_dir / "workspace"
assert workspace.is_dir()
config = CodexConfig(
    cwd=str(workspace),
    env={**os.environ, "LITELLM_API_KEY": api_key},
    config_overrides=(
        'model_provider="litellm"',
        'model_providers.litellm.name="LiteLLM"',
        f"model_providers.litellm.base_url={json.dumps(endpoint)}",
        'model_providers.litellm.env_key="LITELLM_API_KEY"',
        'model_providers.litellm.wire_api="responses"',
        "model_providers.litellm.requires_openai_auth=false",
        "model_providers.litellm.request_max_retries=0",
        "model_providers.litellm.stream_max_retries=0",
        'web_search="disabled"',
    ),
)
print(type(config))

<class 'openai_codex.client.CodexConfig'>


In [4]:
workspace

PosixPath('/home/ian/agent-framework-experiment/codex-sdk/workspace')

### 控制流程圖

本教學分別展示完整結果與串流事件。兩種路徑都透過 `async with` 管理 runtime。

執行下方區塊可顯示圖形。此區塊不會呼叫 LLM。


In [2]:
from IPython.display import HTML, display

display(HTML(r"""
<section style="background:#f8fafc;color:#172b4d;border:1px solid #dce4ef;border-radius:16px;padding:18px;margin:12px 0;font-family:system-ui,sans-serif;overflow-x:auto">
<h3 style="color:#172b4d;margin:0 0 12px;font-size:20px">控制流程圖</h3>
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1080 775" role="img" aria-label="Codex SDK 完整回傳與事件串流的控制流程" style="display:block;width:100%;min-width:760px;height:auto;font-family:'Noto Sans CJK TC','Microsoft JhengHei',system-ui,sans-serif">
 <title>Codex SDK 完整回傳與事件串流的控制流程</title><defs><marker id="codex-sdk-flow-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#64748b"/></marker></defs>
 <rect width="1080" height="775" fill="#f8fafc"/><rect x="385" y="20" width="310" height="68" rx="12" fill="#f0f6ff" stroke="#2563eb" stroke-width="1.5"/><text x="540.0" y="48" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">CodexConfig</text><text x="540.0" y="71" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">設定 LiteLLM Responses API</text><rect x="385" y="130" width="310" height="68" rx="12" fill="#f0f6ff" stroke="#2563eb" stroke-width="1.5"/><text x="540.0" y="158" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">async with AsyncCodex(config)</text><text x="540.0" y="181" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">啟動 runtime；負責資源生命週期</text><rect x="385" y="240" width="310" height="68" rx="12" fill="#f0f6ff" stroke="#2563eb" stroke-width="1.5"/><text x="540.0" y="268" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">await codex.thread_start(...)</text><text x="540.0" y="291" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">取得 AsyncThread</text><rect x="90" y="375" width="350" height="68" rx="12" fill="#f0f6ff" stroke="#087e8b" stroke-width="1.5"/><text x="265.0" y="403" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">await thread.run(str)</text><text x="265.0" y="426" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">等待完整 turn，取得 TurnResult</text><rect x="640" y="375" width="350" height="68" rx="12" fill="#f0f6ff" stroke="#087e8b" stroke-width="1.5"/><text x="815.0" y="403" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">await thread.turn([TextInput(...)])</text><text x="815.0" y="426" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">取得 AsyncTurnHandle</text><rect x="640" y="500" width="350" height="68" rx="12" fill="#f0f6ff" stroke="#087e8b" stroke-width="1.5"/><text x="815.0" y="528" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">async for event in handle.stream()</text><text x="815.0" y="551" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">逐一讀取 Notification</text><rect x="385" y="640" width="310" height="68" rx="12" fill="#f0f6ff" stroke="#2563eb" stroke-width="1.5"/><text x="540.0" y="668" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">離開 async with</text><text x="540.0" y="691" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">關閉 app-server；保留已取得的資料</text><polyline points="540,88 540,130" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-flow-arrow)"/><polyline points="540,198 540,240" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-flow-arrow)"/><polyline points="480,308 480,340 265,340 265,375" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-flow-arrow)"/><text x="270" y="330" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">完整結果</text><polyline points="600,308 600,340 815,340 815,375" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-flow-arrow)"/><text x="815" y="330" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">串流事件</text><polyline points="815,443 815,500" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-flow-arrow)"/><polyline points="265,443 265,674 385,674" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-flow-arrow)"/><polyline points="815,568 815,674 695,674" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-flow-arrow)"/><text x="885" y="615" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">turn/completed</text><text x="30" y="747" text-anchor="start" fill="#52647c" font-size="14" font-weight="400">兩條路徑使用不同的 turn。不要重複消費同一個 handle 的串流。</text></svg>
<p style="color:#52647c;line-height:1.7;margin:10px 0 0;font-size:14px">本教學分別展示完整結果與串流事件。兩種路徑都透過 async with 管理 runtime。</p>
</section>
"""))


## 4. 原生字串輸入 → TurnResult

`thread_start` 建立 thread。`thread.run` 送出一個 turn，等待完成後回傳 `TurnResult`。
`async with` 負責啟動與關閉 app-server。離開區塊後，回傳資料仍可讀取。
本範例使用 `ephemeral=True`，不示範跨程序恢復 thread。

任務直接包含程式碼。本機沙箱限制可能阻擋 shell 讀檔，詳見專案 README。


In [5]:
prompt = (
    "以繁體中文分析這段 Python："
    "def average(values): return sum(values) / len(values)。"
    "指出空清單觸發的例外名稱，並用兩句話提出修正建議。不要執行工具。"
)
print(type(prompt))
print(prompt)

async with AsyncCodex(config) as codex:
    thread = await codex.thread_start(
        model=model_name,
        model_provider="litellm",
        cwd=str(workspace),
        sandbox=Sandbox.read_only,
        approval_mode=ApprovalMode.deny_all,
        ephemeral=True,
    )
    print(type(thread))
    print("thread id：", thread.id)
    result = await thread.run(prompt)

print(type(result))
pprint(result)

<class 'str'>
以繁體中文分析這段 Python：def average(values): return sum(values) / len(values)。指出空清單觸發的例外名稱，並用兩句話提出修正建議。不要執行工具。
<class 'openai_codex.api.AsyncThread'>
thread id： 01a1a21e-efed-7e81-a07d-deaca259cbd5
<class 'openai_codex._run.TurnResult'>
TurnResult(id='01a1a21e-f018-7cf3-948e-5d6056d90742',
           status=<TurnStatus.completed: 'completed'>,
           error=None,
           started_at=1793721298,
           completed_at=1793721307,
           duration_ms=8100,
           final_response='這段函式會先計算 `sum(values)` 再除以 `len(values)`；當 `values` '
                          '是空清單時，`len(values) == 0`，因此會觸發 '
                          '`ZeroDivisionError`。  \n'
                          '\n'
                          '修正建議：先在函式一開始檢查 `values` '
                          '是否為空，若為空就回傳預設值、`None`，或直接拋出更清楚的自訂例外。  \n'
                          '也可以改成先處理空清單再做平均，避免把除以零的錯誤留到執行期才發生。',
           items=[ThreadItem(root=UserMessageThreadItem(client_id=None, content=[UserInput(root=TextUserInput(text

## 5. 讀取 TurnResult 原生欄位

`id` 是 turn ID；它與 thread ID 不同。
`status` 是原生狀態值。`final_response` 是最終文字，也可能為 `None`。
`items` 保存本次 turn 的原生項目。項目種類可能包含訊息或工具操作。
`usage` 是原生用量物件，也可能為 `None`。
`run` 可能因失敗而拋出 SDK 例外；請勿把它理解成固定回傳錯誤字典。


In [6]:
print("id：", result.id)
print("status：", result.status)
print("error：", result.error)
print("duration_ms：", result.duration_ms)
print("final_response：", result.final_response)
print("items 型別：", type(result.items))
pprint(result.items)
for item in result.items:
    print(type(item))
    pprint(item.model_dump(mode="json"))
print("usage 型別：", type(result.usage))
pprint(result.usage)
if result.usage is not None:
    pprint(result.usage.model_dump(mode="json"))

id： 01a1a21e-f018-7cf3-948e-5d6056d90742
status： TurnStatus.completed
error： None
duration_ms： 8100
final_response： 這段函式會先計算 `sum(values)` 再除以 `len(values)`；當 `values` 是空清單時，`len(values) == 0`，因此會觸發 `ZeroDivisionError`。  

修正建議：先在函式一開始檢查 `values` 是否為空，若為空就回傳預設值、`None`，或直接拋出更清楚的自訂例外。  
也可以改成先處理空清單再做平均，避免把除以零的錯誤留到執行期才發生。
items 型別： <class 'list'>
[ThreadItem(root=UserMessageThreadItem(client_id=None, content=[UserInput(root=TextUserInput(text='以繁體中文分析這段 Python：def average(values): return sum(values) / len(values)。指出空清單觸發的例外名稱，並用兩句話提出修正建議。不要執行工具。', text_elements=[], type='text'))], id='01a1a21e-f17d-7341-b374-912370dfac2d', type='userMessage')),
 ThreadItem(root=ReasoningThreadItem(content=[], id='rs_044badb62434332f016aea03d8cb1c87d2af4bf0576e37a5df', summary=[], type='reasoning')),
 ThreadItem(root=AgentMessageThreadItem(delivery=None, id='msg_044badb62434332f016aea03da19bc87d2bb9b1eed0e637c9a', memory_citation=None, phase=<MessagePhase.final_answer: 'final_answer'>, questions=None, text

### 循序圖

`thread.run` 會收集事件並回傳結果。下一節的 `thread.turn` 與 `handle.stream` 讓你直接讀取通知。

執行下方區塊可顯示圖形。此區塊不會呼叫 LLM。


In [3]:
from IPython.display import HTML, display

display(HTML(r"""
<section style="background:#f8fafc;color:#172b4d;border:1px solid #dce4ef;border-radius:16px;padding:18px;margin:12px 0;font-family:system-ui,sans-serif;overflow-x:auto">
<h3 style="color:#172b4d;margin:0 0 12px;font-size:20px">循序圖</h3>
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1080 454" role="img" aria-label="Codex Python SDK 到本機 runtime 與 LiteLLM 的循序圖" style="display:block;width:100%;min-width:760px;height:auto;font-family:'Noto Sans CJK TC','Microsoft JhengHei',system-ui,sans-serif">
 <title>Codex Python SDK 到本機 runtime 與 LiteLLM 的循序圖</title><defs><marker id="codex-sdk-sequence-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#64748b"/></marker></defs>
 <rect width="1080" height="454" fill="#f8fafc"/><rect x="16.0" y="15" width="188" height="62" rx="10" fill="#fff" stroke="#2563eb"/><text x="110.0" y="41" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">Notebook</text><text x="110.0" y="62" text-anchor="middle" fill="#52647c" font-size="11" font-weight="400">使用者程式</text><line x1="110.0" y1="77" x2="110.0" y2="404" stroke="#b6c4d6" stroke-dasharray="5 5"/><rect x="302.6666666666667" y="15" width="188" height="62" rx="10" fill="#fff" stroke="#2563eb"/><text x="396.6666666666667" y="41" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">Python SDK</text><text x="396.6666666666667" y="62" text-anchor="middle" fill="#52647c" font-size="11" font-weight="400">openai_codex</text><line x1="396.6666666666667" y1="77" x2="396.6666666666667" y2="404" stroke="#b6c4d6" stroke-dasharray="5 5"/><rect x="589.3333333333334" y="15" width="188" height="62" rx="10" fill="#fff" stroke="#2563eb"/><text x="683.3333333333334" y="41" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">Codex app-server</text><text x="683.3333333333334" y="62" text-anchor="middle" fill="#52647c" font-size="11" font-weight="400">本機子程序</text><line x1="683.3333333333334" y1="77" x2="683.3333333333334" y2="404" stroke="#b6c4d6" stroke-dasharray="5 5"/><rect x="876.0" y="15" width="188" height="62" rx="10" fill="#fff" stroke="#2563eb"/><text x="970.0" y="41" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">LiteLLM</text><text x="970.0" y="62" text-anchor="middle" fill="#52647c" font-size="11" font-weight="400">Responses API</text><line x1="970.0" y1="77" x2="970.0" y2="404" stroke="#b6c4d6" stroke-dasharray="5 5"/><polyline points="110.0,115 396.6666666666667,115" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-sequence-arrow)"/><text x="253.33333333333334" y="105" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">1. thread.run(prompt)</text><polyline points="396.6666666666667,164 683.3333333333334,164" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-sequence-arrow)"/><text x="540.0" y="154" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">2. JSON-RPC：turn/start</text><polyline points="683.3333333333334,213 970.0,213" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-sequence-arrow)"/><text x="826.6666666666667" y="203" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">3. POST /responses</text><polyline points="970.0,262 683.3333333333334,262" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-sequence-arrow)" stroke-dasharray="6 4"/><text x="826.6666666666667" y="252" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">4. Responses 串流事件</text><polyline points="683.3333333333334,311 396.6666666666667,311" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-sequence-arrow)" stroke-dasharray="6 4"/><text x="540.0" y="301" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">5. item/*、turn/completed 通知</text><polyline points="396.6666666666667,360 110.0,360" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#codex-sdk-sequence-arrow)" stroke-dasharray="6 4"/><text x="253.33333333333334" y="350" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">6. 彙整成 TurnResult</text><text x="30" y="432" text-anchor="start" fill="#52647c" font-size="13" font-weight="400">實線：呼叫或請求　　虛線：回傳資料或事件</text></svg>
<p style="color:#52647c;line-height:1.7;margin:10px 0 0;font-size:14px">thread.run 會收集事件並回傳結果。下一節的 thread.turn 與 handle.stream 讓你直接讀取通知。</p>
</section>
"""))


## 6. 原生 TextInput → 事件串流

除了字串，SDK 也接受 `TextInput` 項目的清單。
`thread.turn` 回傳 handle。使用 `handle.stream()` 逐一接收原生通知。
通知的 `method` 表示事件種類，`payload` 是事件資料。
常見事件包含 `turn/started`、文字增量與 `turn/completed`。
事件順序與數量依 runtime 與模型而變。

此區塊建立另一個 thread。串流只消費一次，不再對同一個 handle 呼叫 `run()`。


In [7]:
turn_input = [TextInput(text="用一句繁體中文說明 Python 清單。不要執行工具。")]
pprint(turn_input)
print(type(turn_input[0]))

notifications = []
async with AsyncCodex(config) as codex:
    thread = await codex.thread_start(
        model=model_name,
        model_provider="litellm",
        cwd=str(workspace),
        sandbox=Sandbox.read_only,
        approval_mode=ApprovalMode.deny_all,
        ephemeral=True,
    )
    handle = await thread.turn(turn_input)
    print(type(handle))
    async for notification in handle.stream():
        notifications.append(notification)
        print(type(notification))
        pprint(notification)

[TextInput(text='用一句繁體中文說明 Python 清單。不要執行工具。')]
<class 'openai_codex._inputs.TextInput'>
<class 'openai_codex.api.AsyncTurnHandle'>
<class 'openai_codex.models.Notification'>
Notification(method='turn/started',
             payload=TurnStartedNotification(thread_id='01a1a221-41c4-7601-80a6-84dff8448b5f', turn=Turn(completed_at=None, duration_ms=None, error=None, id='01a1a221-41f0-7c23-a01c-ed1c1a2123f6', items=[], items_view=<TurnItemsView.not_loaded: 'notLoaded'>, root_turn_id='01a1a221-41f0-7c23-a01c-ed1c1a2123f6', started_at=1793721450, status=<TurnStatus.in_progress: 'inProgress'>)))
<class 'openai_codex.models.Notification'>
Notification(method='item/started',
             payload=ItemStartedNotification(item=ThreadItem(root=UserMessageThreadItem(client_id=None, content=[UserInput(root=TextUserInput(text='用一句繁體中文說明 Python 清單。不要執行工具。', text_elements=[], type='text'))], id='01a1a221-453f-76e2-905f-e9a39449081b', type='userMessage')), started_at_ms=1793721451839, thread_id='01a1a221-

## 7. 檢查通知與完成事件

保留完整 `notifications` 清單，再從中查看事件。
串流的完成通知是 `TurnCompletedNotification`，不是 `TurnResult`。
`payload.turn` 提供該 turn 的狀態與項目。請檢查 `status` 與 `error`。

`payload.turn.items_view` 表示項目檢視方式。若值為 `summary`，完成事件中的 `items` 可能只是摘要。
需要觀察逐項事件時，請查看前一區塊保留的 `notifications`。


In [7]:
print("事件方法：", [event.method for event in notifications])
completed = next(event for event in notifications if event.method == "turn/completed")
print(type(completed.payload))
pprint(completed.payload)
print("turn status：", completed.payload.turn.status)
print("turn error：", completed.payload.turn.error)
pprint(completed.payload.model_dump(mode="json"))

事件方法： ['turn/started', 'item/started', 'item/completed', 'item/started', 'item/completed', 'item/started', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/agentMessage/delta', 'item/completed', 'thread/tokenUsage/updated', 'turn/completed']
<class 'openai_codex.generated.v2_all.TurnCompletedNotification'>
TurnCompletedNotification(thread_id='01a1a209-04cb-7192-aed0

## 官方來源與版本

- [Codex SDK Quickstart](https://learn.chatgpt.com/docs/codex-sdk)：`AsyncCodex`、thread 與 run。
- [官方 Python SDK 範例](https://github.com/openai/codex/tree/main/sdk/python/examples)：同步與非同步範例。
- [Codex 供應者設定](https://learn.chatgpt.com/docs/config-file/config-advanced)：LiteLLM 自訂供應者。

型別與欄位已核對本專案鎖定的 `openai-codex` 版本。
模型回覆與用量會隨執行改變。
